# Notebook 3: Memoria y Persistencia con LangGraph

## ¿De qué trata este notebook?

Los grafos de los notebooks 1 y 2 tienen un problema fundamental: **no recuerdan nada**. Cada vez que ejecutas `invoke()`, el grafo empieza desde cero. No sabe nada de conversaciones anteriores.

### La analogía del médico de cabecera

Imagina un médico que al llegar cada paciente no tiene ni idea de quién es. No recuerda que ya vino la semana pasada, no sabe sus alergias, ni su historial. Cada visita empieza de cero.

Eso es lo que ocurre sin memoria. Con memoria, el médico tiene la ficha del paciente: sabe quién es, qué le dijiste antes, y puede continuar exactamente desde donde lo dejó.

### ¿Qué es un checkpoint?

Un checkpoint es una **foto del estado** del grafo en un momento dado. LangGraph guarda automáticamente una foto después de cada nodo. Si retomas la conversación más tarde, carga la última foto y continúa desde ahí.

### Conceptos nuevos en este notebook

| Concepto | ¿Qué es? |
|----------|---------|
| **MemorySaver** | Guarda fotos (checkpoints) en RAM. Se pierden al reiniciar |
| **SqliteSaver** | Guarda fotos en un archivo de base de datos. Sobreviven al reinicio |
| **thread_id** | El número de expediente que identifica cada conversación |
| **add_messages** | Un reducer que acumula mensajes en lugar de reemplazarlos |
| **Reducer** | Una regla que dice cómo combinar el valor nuevo con el existente |
| **config** | El diccionario que le dice al grafo en qué conversación estamos |

---
## 1. Instalación y configuración

In [5]:
# %pip install -qU langgraph langchain-google-genai langchain-core

In [6]:
from dotenv import load_dotenv
import os

load_dotenv(dotenv_path="C:/Users/Oscar/OneDrive - FM4/Escritorio/EVOLVE/Data Science/EVOLVE/Victor_Prior_IA_Generativa/Proyecto_Modulo_IAGen/.env")
API_KEY = os.getenv("GEMINI_API_KEY_001")

# API key de Gemini
# API_KEY = userdata.get('GEMINI_API_KEY')

In [7]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    temperature=0.7,
    api_key = API_KEY
)
print("Modelo listo:", llm.model)

Modelo listo: gemini-2.5-flash-lite


---
## 2. El problema: sin memoria, el chatbot olvida todo

### Demostración del problema

Aquí vemos el problema en acción: dos llamadas consecutivas al LLM donde la segunda no sabe nada de la primera.

- `llm.invoke("Hola, mi nombre es Carlos")`: el modelo saluda
- `llm.invoke("¿Cómo me llamo?")`: el modelo no tiene ni idea — cada llamada es completamente independiente

La solución estará en los siguientes pasos.

In [8]:
# Sin memoria: cada llamada es independiente
from langchain_core.messages import HumanMessage, AIMessage

r1 = llm.invoke("Hola, mi nombre es Carlos")
print("Turno 1:", r1.content)

r2 = llm.invoke("¿Cómo me llamo?")
print("Turno 2:", r2.content)  # ← No sabe que se llama Carlos

Turno 1: ¡Hola, Carlos! Encantado de conocerte. ¿En qué puedo ayudarte hoy?
Turno 2: Lo siento, no tengo acceso a información personal como tu nombre. Soy un modelo de lenguaje grande, entrenado por Google.


---
## 3. La solución: estado con lista de mensajes + reducers

### ¿Qué es un reducer?

Recuerda que en los notebooks anteriores, cuando un nodo devuelve un valor, reemplaza el campo existente en el estado. Pero para el historial de mensajes eso sería un desastre: cada respuesta borraría la anterior.

Un **reducer** es una regla que dice cómo combinar el valor nuevo con el existente:
- **Sin reducer (default):** `estado['campo'] = valor_nuevo` → **reemplaza**
- **Con `add_messages`:** `estado['mensajes'] = estado['mensajes'] + nuevos_mensajes` → **acumula**

La sintaxis `Annotated[list[BaseMessage], add_messages]` le dice a LangGraph: "este campo es una lista de mensajes y cuando un nodo devuelva nuevos mensajes, añádelos a la lista en lugar de reemplazarla".

**Lo que entra:** nada (es solo una definición de estructura)  
**Lo que produce:** un estado que acumula mensajes a lo largo de la conversación

In [9]:
from typing import Annotated, TypedDict
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage

class EstadoChat(TypedDict):
    # 'add_messages' es el reducer: AÑADE nuevos mensajes a la lista
    # en lugar de reemplazar toda la lista.
    # Sin reducer: estado['mensajes'] = nuevos_mensajes  (reemplaza)
    # Con reducer:  estado['mensajes'] += nuevos_mensajes (acumula)
    mensajes: Annotated[list[BaseMessage], add_messages]

print("Estado con reducer 'add_messages' definido")
print("Tipo de mensajes:", EstadoChat.__annotations__["mensajes"])

Estado con reducer 'add_messages' definido
Tipo de mensajes: typing.Annotated[list[langchain_core.messages.base.BaseMessage], <function _add_messages_wrapper.<locals>._add_messages at 0x0000019B4B4C5EE0>]


---
## 4. Nodo del chatbot

### ¿Cómo funciona la memoria dentro del nodo?

La clave está en que este nodo recibe en `estado["mensajes"]` **todos los mensajes de la conversación** hasta ese momento, no solo el último.

Entonces, al llamar al LLM, le pasamos todo el historial. El modelo puede leer lo que se dijo antes y responder con coherencia.

El nodo devuelve solo el mensaje nuevo (la respuesta). El reducer `add_messages` se encargará de añadirlo al historial existente.

**`SystemMessage`**: es el mensaje que establece la personalidad del chatbot. Solo lo ve el modelo, no el usuario. Va al principio de la lista de mensajes en cada llamada.

In [10]:
from langchain_core.messages import SystemMessage

SYSTEM_PROMPT = """Eres un asistente amigable que recuerda todo lo que el usuario te ha dicho
en la conversación. Personaliza tus respuestas con la información que el usuario ha compartido."""

def nodo_chatbot(estado: EstadoChat) -> dict:
    """
    Recibe el historial completo de mensajes y genera una respuesta.
    El historial incluye TODOS los turnos anteriores gracias al reducer.
    """
    # Preparar mensajes con sistema prompt
    mensajes_completos = [SystemMessage(content=SYSTEM_PROMPT)] + estado["mensajes"]
    
    respuesta = llm.invoke(mensajes_completos)
    
    # Retornamos solo el nuevo mensaje; el reducer lo añadirá al historial
    return {"mensajes": [respuesta]}

print("Nodo chatbot definido")

Nodo chatbot definido


---
## 5. Construir el grafo con MemorySaver

### ¿Qué hace `MemorySaver`?

`MemorySaver` es el archivador que guarda una foto del estado después de cada nodo. Funciona en RAM (la memoria del ordenador mientras está encendido).

**Limitación importante:** cuando el proceso Python termina o reinicias el kernel, los datos desaparecen. Para guardar datos de forma permanente, usa `SqliteSaver` (ver sección 9).

La diferencia clave respecto a los notebooks anteriores está en `grafo.compile(checkpointer=memoria)`. Ese parámetro extra es lo que activa el sistema de memoria.

In [11]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

# Construir el grafo
grafo = StateGraph(EstadoChat)
grafo.add_node("chatbot", nodo_chatbot)
grafo.add_edge(START, "chatbot")
grafo.add_edge("chatbot", END)

# MemorySaver: guarda checkpoints en memoria (RAM)
# Para producción: usar SqliteSaver o PostgresSaver
memoria = MemorySaver()

# Compilar CON el checkpointer
app = grafo.compile(checkpointer=memoria)

print("✅ Chatbot con memoria compilado")

✅ Chatbot con memoria compilado


---
## 6. El thread_id: clave para múltiples conversaciones

### ¿Qué es el thread_id?

El `thread_id` es el número de expediente de una conversación. LangGraph lo usa para:
- Guardar el historial asociado a esa conversación
- Recuperarlo en la siguiente llamada
- Mantener conversaciones completamente separadas entre sí

**El `config`** es el diccionario que le pasas al `invoke()` para decirle en qué conversación estamos. Es obligatorio cuando usas checkpointer.

**Lo que entra en `chatear()`:** un mensaje de texto y el thread_id de la conversación  
**Lo que sale:** el texto de la respuesta del modelo

In [12]:
from langchain_core.messages import HumanMessage

def chatear(app, mensaje: str, thread_id: str):
    """Envía un mensaje al chatbot dentro de un thread específico."""
    # El config es OBLIGATORIO cuando usamos checkpointer
    config = {"configurable": {"thread_id": thread_id}}
    
    estado_entrada = {"mensajes": [HumanMessage(content=mensaje)]}
    
    resultado = app.invoke(estado_entrada, config=config)
    
    # El último mensaje es la respuesta del AI
    respuesta = resultado["mensajes"][-1].content
    return respuesta

print("Función de chat lista")

Función de chat lista


In [13]:
# Conversación 1: Usuario 'Carlos'
print("=" * 60)
print("CONVERSACIÓN DE CARLOS (thread: carlos-001)")
print("=" * 60)

r = chatear(app, "Hola, me llamo Carlos y soy programador de Python.", "carlos-001")
print(f"Usuario: Hola, me llamo Carlos y soy programador de Python.")
print(f"Bot: {r}\n")

r = chatear(app, "¿Cuál es mi lenguaje de programación favorito?", "carlos-001")
print(f"Usuario: ¿Cuál es mi lenguaje de programación favorito?")
print(f"Bot: {r}\n")

r = chatear(app, "Tengo 5 años de experiencia. ¿Me darías un consejo para mejorar?", "carlos-001")
print(f"Usuario: Tengo 5 años de experiencia. ¿Me darías un consejo para mejorar?")
print(f"Bot: {r}\n")

CONVERSACIÓN DE CARLOS (thread: carlos-001)
Usuario: Hola, me llamo Carlos y soy programador de Python.
Bot: ¡Hola Carlos! Encantado de conocerte. Es genial saber que eres programador de Python. Es un lenguaje muy versátil y popular.

¿Hay algo en particular en lo que pueda ayudarte hoy, Carlos? Quizás necesites ayuda con algún código, buscas información sobre alguna librería de Python, o simplemente quieres charlar sobre algún tema relacionado con la programación.

Usuario: ¿Cuál es mi lenguaje de programación favorito?
Bot: Carlos, según lo que me dijiste antes, tu lenguaje de programación favorito es **Python**.

Usuario: Tengo 5 años de experiencia. ¿Me darías un consejo para mejorar?
Bot: ¡Claro que sí, Carlos! Con 5 años de experiencia en Python, ya tienes una base sólida. Para seguir mejorando, te daría este consejo:

**Profundiza en las herramientas y patrones avanzados de Python, y considera contribuir a proyectos de código abierto.**

Déjame explicarte por qué, pensando en tu

In [14]:
# Conversación 2: Usuario 'Ana' (completamente independiente)
print("=" * 60)
print("CONVERSACIÓN DE ANA (thread: ana-001)")
print("=" * 60)

r = chatear(app, "Hola, soy Ana y trabajo en diseño gráfico.", "ana-001")
print(f"Usuario: Hola, soy Ana y trabajo en diseño gráfico.")
print(f"Bot: {r}\n")

r = chatear(app, "¿A qué me dedico?", "ana-001")
print(f"Usuario: ¿A qué me dedico?")
print(f"Bot: {r}\n")

CONVERSACIÓN DE ANA (thread: ana-001)
Usuario: Hola, soy Ana y trabajo en diseño gráfico.
Bot: ¡Hola, Ana! Es un placer conocerte. ¡Diseño gráfico! Me parece un campo fascinante.  Entonces, tú eres la que le da vida a las ideas con imágenes y colores, ¿verdad?  ¿Hay algún tipo de diseño gráfico que te apasione especialmente?

Usuario: ¿A qué me dedico?
Bot: Ana, según me dijiste hace un momento, **trabajas en diseño gráfico**. ¡Así que te dedicas a crear todo tipo de elementos visuales!



In [15]:
# Volvemos a Carlos: ¿recuerda quién es?
print("=" * 60)
print("DE VUELTA CON CARLOS (thread: carlos-001)")
print("=" * 60)

r = chatear(app, "¿Cómo me llamo y qué hago?", "carlos-001")
print(f"Usuario: ¿Cómo me llamo y qué hago?")
print(f"Bot: {r}")

DE VUELTA CON CARLOS (thread: carlos-001)
Usuario: ¿Cómo me llamo y qué hago?
Bot: Carlos, tú eres **programador de Python** y tienes **5 años de experiencia** en este campo.


---
## 7. Inspeccionar el estado guardado

### ¿Para qué sirve inspeccionar el estado?

`app.get_state(config)` devuelve el estado actual de una conversación sin ejecutar ningún nodo. Es de solo lectura.

Útil para:
- Debuggear: ver exactamente qué mensajes tiene guardados el chatbot
- Auditoría: comprobar el historial de una conversación
- Monitoreo: saber cuántos turnos lleva una conversación

El estado de Carlos tiene 6 mensajes porque tuvo 3 turnos (3 mensajes del usuario + 3 respuestas del bot).

In [16]:
# Ver el estado actual del thread de Carlos
config_carlos = {"configurable": {"thread_id": "carlos-001"}}
snapshot = app.get_state(config_carlos)

print("Estado actual del thread 'carlos-001':")
print(f"Número de mensajes en historial: {len(snapshot.values['mensajes'])}")
print("\nHistorial completo:")
for i, msg in enumerate(snapshot.values["mensajes"]):
    tipo = "Usuario" if msg.__class__.__name__ == "HumanMessage" else "Bot"
    print(f"  [{i+1}] {tipo}: {msg.content[:80]}..." if len(msg.content) > 80 else f"  [{i+1}] {tipo}: {msg.content}")

Estado actual del thread 'carlos-001':
Número de mensajes en historial: 8

Historial completo:
  [1] Usuario: Hola, me llamo Carlos y soy programador de Python.
  [2] Bot: ¡Hola Carlos! Encantado de conocerte. Es genial saber que eres programador de Py...
  [3] Usuario: ¿Cuál es mi lenguaje de programación favorito?
  [4] Bot: Carlos, según lo que me dijiste antes, tu lenguaje de programación favorito es *...
  [5] Usuario: Tengo 5 años de experiencia. ¿Me darías un consejo para mejorar?
  [6] Bot: ¡Claro que sí, Carlos! Con 5 años de experiencia en Python, ya tienes una base s...
  [7] Usuario: ¿Cómo me llamo y qué hago?
  [8] Bot: Carlos, tú eres **programador de Python** y tienes **5 años de experiencia** en ...


---
## 8. Ver el historial de checkpoints

### ¿Qué es el historial de checkpoints?

LangGraph guarda una foto del estado después de **cada nodo** de cada ejecución. El historial de checkpoints es la lista de todas esas fotos, ordenadas de la más reciente a la más antigua.

Esto permite hacer "time travel": retomar la conversación desde cualquier punto anterior, no solo desde el último.

In [17]:
# Ver todos los checkpoints guardados para un thread
historial = list(app.get_state_history(config_carlos))

print(f"Total de checkpoints para 'carlos-001': {len(historial)}")
print("\nCheckpoints (del más reciente al más antiguo):")
for i, checkpoint in enumerate(historial):
    n_msgs = len(checkpoint.values.get("mensajes", []))
    ts = checkpoint.metadata.get("step", "?")
    print(f"  Checkpoint {i+1}: step={ts}, mensajes={n_msgs}")

Total de checkpoints para 'carlos-001': 12

Checkpoints (del más reciente al más antiguo):
  Checkpoint 1: step=10, mensajes=8
  Checkpoint 2: step=9, mensajes=7
  Checkpoint 3: step=8, mensajes=6
  Checkpoint 4: step=7, mensajes=6
  Checkpoint 5: step=6, mensajes=5
  Checkpoint 6: step=5, mensajes=4
  Checkpoint 7: step=4, mensajes=4
  Checkpoint 8: step=3, mensajes=3
  Checkpoint 9: step=2, mensajes=2
  Checkpoint 10: step=1, mensajes=2
  Checkpoint 11: step=0, mensajes=1
  Checkpoint 12: step=-1, mensajes=0


---
## 9. Bonus: SqliteSaver para persistencia real

### ¿Cuándo necesitas SqliteSaver?

`MemorySaver` guarda los datos en RAM. Cuando el proceso termina (cierras el notebook, reinicias el kernel), los datos desaparecen.

`SqliteSaver` guarda los checkpoints en un archivo `.db` en tu disco. Los datos **sobreviven** entre sesiones: la próxima vez que ejecutes el notebook, la conversación continuará exactamente desde donde la dejó.

| | MemorySaver | SqliteSaver |
|-|-------------|-------------|
| Dónde guarda | RAM | Archivo .db en disco |
| Cuándo se pierde | Al reiniciar el proceso | Nunca (hasta que borres el archivo) |
| Uso típico | Desarrollo, pruebas | Producción, demos persistentes |

El `with SqliteSaver.from_conn_string(...)` es un gestor de contexto: abre la base de datos al entrar y la cierra correctamente al salir.

In [18]:
%pip install -qU langgraph-checkpoint-sqlite

Note: you may need to restart the kernel to use updated packages.


In [19]:
from langgraph.checkpoint.sqlite import SqliteSaver

# Los datos se guardan en un archivo .db local
with SqliteSaver.from_conn_string("chatbot_historia.db") as db_memory:
    app_sqlite = grafo.compile(checkpointer=db_memory)
    
    config = {"configurable": {"thread_id": "persistente-001"}}
    
    # Esta conversación sobrevive entre reinicios del proceso
    r = app_sqlite.invoke(
        {"mensajes": [HumanMessage(content="Hola, soy una conversación persistente.")]},
        config=config
    )
    print("Bot:", r["mensajes"][-1].content)

print("\n✅ Conversación guardada en chatbot_historia.db")
print("Si ejecutas este notebook de nuevo, la conversación continuará desde aquí.")

Bot: ¡Hola, conversación persistente! Es un placer conocerte. Me alegra que hayas decidido interactuar conmigo. ¿En qué puedo ayudarte hoy?

✅ Conversación guardada en chatbot_historia.db
Si ejecutas este notebook de nuevo, la conversación continuará desde aquí.


---
## 10. Ejercicios propuestos

1. **Límite de memoria**: Modifica el nodo chatbot para que solo envíe los últimos 6 mensajes al LLM (ventana deslizante), aunque el historial completo se guarde en el estado.

2. **Resumen de conversación**: Añade un segundo nodo que se active cada 10 mensajes y resuma la conversación, almacenando el resumen en un campo `resumen` del estado.

3. **Múltiples usuarios**: Crea una función que genere `thread_id` únicos por usuario usando UUID, y simula 3 usuarios chateando en paralelo.

## Resumen

| Concepto | Descripción |
|---|---|
| `MemorySaver` | Guarda checkpoints en RAM (en proceso) |
| `SqliteSaver` | Guarda checkpoints en SQLite (entre sesiones) |
| `thread_id` | Identifica una conversación única |
| `add_messages` | Reducer que acumula mensajes en lugar de reemplazarlos |
| `get_state()` | Lee el estado actual de un thread |
| `get_state_history()` | Lista todos los checkpoints de un thread |
| `config` | Diccionario con `{"configurable": {"thread_id": ...}}` requerido para checkpointing |